# E3 — GloVe + LSTM + Dropout

**E3 — GloVe + LSTM + Dropout/recurrent_dropout regularization.**

In [2]:
import os, re, time, json, pickle
import numpy as np
import pandas as pd

SEED = 42
DATA_PATH = "../data/IMDB Dataset.csv"
RESULTS_DIR = "../results"
TOKENIZER_PATH = "../results/tokenizer.pkl"
VOCAB_SIZE = 10000
EMBED_DIM = 100
SAMPLE_SIZE = 15000   # <-- subsample for faster training

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["review", "sentiment"])
df["review"] = df["review"].apply(lambda t: re.sub(r"<br\s*/?>", " ", str(t)))
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
assert df["label"].isna().sum() == 0, "Unexpected sentiment values — check the column."

# Subsample BEFORE splitting, so train/test shrink together and stay balanced
df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

X = df["review"].astype(str).to_numpy()
y = df["label"].to_numpy(dtype=int)

from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)
print(f"Train: {len(X_train_text)}  Test: {len(X_test_text)}")

Train: 12000  Test: 3000


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(SEED)
np.random.seed(SEED)
MAX_LEN = 200
BATCH_SIZE = 64
EPOCHS = 15
DROPOUT = 0.3
RECURRENT_DROPOUT = 0.2

I0000 00:00:1788182422.154260   96345 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788182422.209874   96345 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788182423.442552   96345 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Reuse the SAME tokenizer across every notebook (loads from disk if a
# previous notebook already built one) so all experiments share one vocab —
# that's what makes comparing accuracy across them fair.
if os.path.exists(TOKENIZER_PATH):
    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)
    print("Loaded existing tokenizer.")
else:
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train_text)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(TOKENIZER_PATH, "wb") as f:
        pickle.dump(tokenizer, f)
    print("Built and saved new tokenizer.")

Loaded existing tokenizer.


In [5]:
GLOVE_PATH = "../data/glove.6B.100d.txt"
if not os.path.exists(GLOVE_PATH):
    raise FileNotFoundError(
        f"{GLOVE_PATH} not found. Download glove.6B.zip from "
        "https://nlp.stanford.edu/projects/glove/ and unzip into data/."
    )

embeddings_index = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        values = line.split()
        embeddings_index[values[0]] = np.asarray(values[1:], dtype="float32")

embedding_matrix = np.random.normal(scale=0.1, size=(VOCAB_SIZE, EMBED_DIM)).astype("float32")
hits = 0
for word, idx in tokenizer.word_index.items():
    if idx >= VOCAB_SIZE:
        continue
    if word in embeddings_index:
        embedding_matrix[idx] = embeddings_index[word]
        hits += 1
print(f"GloVe coverage: {hits}/{VOCAB_SIZE} ({hits/VOCAB_SIZE:.1%})")

GloVe coverage: 9812/10000 (98.1%)


In [6]:
x_train = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN)
x_test = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=MAX_LEN)

model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], input_length=MAX_LEN, trainable=True),
    LSTM(64, dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT),
    Dropout(DROPOUT),
    Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

/home/manikya/nlp_project/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1788182432.531100   96345 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 1,000,000 (3.81 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
start = time.time()
history = model.fit(x_train, y_train, validation_split=0.1,
                     batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=2,
                     callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
train_time = time.time() - start

Epoch 1/15


169/169 - 183s - 1s/step - accuracy: 0.5979 - loss: 0.6569 - val_accuracy: 0.7392 - val_loss: 0.5409
Epoch 2/15
169/169 - 182s - 1s/step - accuracy: 0.7506 - loss: 0.5137 - val_accuracy: 0.7808 - val_loss: 0.4575
Epoch 3/15
169/169 - 187s - 1s/step - accuracy: 0.8198 - loss: 0.4081 - val_accuracy: 0.8467 - val_loss: 0.3635
Epoch 4/15
169/169 - 181s - 1s/step - accuracy: 0.8696 - loss: 0.3184 - val_accuracy: 0.8667 - val_loss: 0.3309
Epoch 5/15
169/169 - 181s - 1s/step - accuracy: 0.8942 - loss: 0.2633 - val_accuracy: 0.8675 - val_loss: 0.3512
Epoch 6/15
169/169 - 222s - 1s/step - accuracy: 0.9165 - loss: 0.2185 - val_accuracy: 0.8717 - val_loss: 0.3494
Epoch 7/15
169/169 - 325s - 2s/step - accuracy: 0.9292 - loss: 0.1899 - val_accuracy: 0.8708 - val_loss: 0.3797


In [8]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

probs = model.predict(x_test, batch_size=BATCH_SIZE).ravel()
preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="binary")
auc = roc_auc_score(y_test, probs)

row = {
    "run_name": "E3_glove_lstm_dropout",
    "accuracy": round(acc, 4), "precision": round(prec, 4), "recall": round(rec, 4),
    "f1": round(f1, 4), "roc_auc": round(auc, 4),
    "train_time_sec": round(train_time, 1),
    "epochs_run": len(history.history["loss"]),
    "params": model.count_params(),
    "max_len": MAX_LEN, "embeddings": "glove", "dropout": DROPOUT, "recurrent_dropout": RECURRENT_DROPOUT,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, "E3_glove_lstm_dropout.csv")
pd.DataFrame([row]).to_csv(out_path, index=False)
print(f"Saved {out_path}")
print(json.dumps(row, indent=2))

47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 416ms/step
Saved ../results/E3_glove_lstm_dropout.csv
{
  "run_name": "E3_glove_lstm_dropout",
  "accuracy": 0.8673,
  "precision": 0.859,
  "recall": 0.8827,
  "f1": 0.8707,
  "roc_auc": 0.9384,
  "train_time_sec": 1462.9,
  "epochs_run": 7,
  "params": 1042305,
  "max_len": 200,
  "embeddings": "glove",
  "dropout": 0.3,
  "recurrent_dropout": 0.2
}
